# 案例 07 · 会话临时文件耗尽 inode，磁盘“有空间却写不进去”

> 沙箱实验：真实进程 + 阿里云 OpenAPI 模拟（响应带 `Mock: true`）。
> 云产品字段/命令与生产一致，资源规格与账单为模拟值。

## 场景

游客模式的草稿会话文件没有 TTL 清理，/data/sessions 堆积近 2 万个空文件，`df -h` 显示仍有空间，但写入报 `Errno 28 (inode exhausted)`。

## 练习目标

- 用 `df -i` 识别 inode 耗尽
- 用 `find | wc -l` 定位高密度小文件目录
- 按 TTL 清理并验证写入恢复
- 用 `inode_guard.py` 完成自动化处置

## 前置

- 仓库根目录执行 `python -m venv .venv && .venv/bin/pip install -r requirements.txt`
- 使用该 venv 的 Python kernel 打开本 Notebook；或运行 `tools/run_all_labs.py` 批量执行


In [ ]:
# 环境准备：定位仓库根目录并加载沙箱
import json, os, signal, ssl, socket, sys, time, gzip, shutil, urllib.parse, urllib.request
from pathlib import Path

REPO = Path.cwd().resolve()
while not (REPO / "labs" / "common" / "xhs_lab.py").exists():
    if REPO.parent == REPO:
        raise RuntimeError("请在 xhs-backend-cloud-ops-lab 仓库内运行本 Notebook")
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from labs.common import Lab

lab = Lab("case-07-inode")
lab.reset(start_api=True)

lab.summary()


## 步骤 1 · 异常重现

下面注入一个可复现的真实故障，观察它对业务的影响。

In [ ]:
lab.inject_inode_exhaustion()
_, files = lab.disk_usage()
print(f"已注入：{files} 个文件 / inode 上限 {lab.inode_limit}")


## 步骤 2 · 取证

按“入口 → 主机 → 进程 → 依赖 → 日志”的分层顺序收集证据，不要一上来就重启。

**空间使用率**

In [ ]:
print(lab.sh("df -h /data"))

**inode 使用率**

In [ ]:
print(lab.sh("df -i /data"))

**会话文件数**

In [ ]:
print(lab.sh("find /data/sessions -type f | wc -l"))

**会话目录大小**

In [ ]:
print(lab.sh("du -sh /data/sessions"))

## 步骤 3 · 判断

把证据收敛成一句结论，决定止血动作。

In [ ]:
used, files = lab.disk_usage()
print(f"空间: {used / lab.quota_mb * 100:.0f}%；inode: {files / lab.inode_limit * 100:.0f}%")
print("判断：空间正常但 inode 接近 100% → 小文件耗尽 inode，不是磁盘容量问题")


## 步骤 4 · 手工处置（先止血）

执行最小必要的变更，优先恢复业务，再考虑根治。

In [ ]:
cutoff = time.time() - 86400
sessions = lab.data / "sessions"
removed = 0
for f in sessions.glob("*.tmp"):
    if f.stat().st_mtime < cutoff:
        f.unlink()
        removed += 1

# 兜底：仍高于 70% 时按最旧 mtime 继续清理到 40%
_, files = lab.disk_usage()
if files / lab.inode_limit > 0.7:
    for f in sorted(sessions.glob("*.tmp"), key=lambda p: p.stat().st_mtime)[:max(files - int(lab.inode_limit * 0.4), 0)]:
        f.unlink(missing_ok=True)

_, files = lab.disk_usage()
print(f"清理 {removed} 个超期文件；inode: {files}/{lab.inode_limit} ({files / lab.inode_limit * 100:.0f}%)")


## 步骤 5 · 验证

用可执行的断言确认恢复，而不是“看起来好了”。

In [ ]:
lab.check('inode < 70%', bool(lab.disk_usage()[1] / lab.inode_limit < 0.7))
lab.check('服务健康', bool('HTTP_CODE=200' in lab.curl(lab.api + '/healthz')))

## 步骤 6 · 自动化沉淀

脚本读取 `df -i`，按 24h TTL 清理会话文件；若仍高于水位，则按 mtime 清理到 40%，并校验 inode 与业务可用性。

下面重置沙箱、重新注入故障，然后直接调用生产可用的自动化脚本（`--apply`；缺省为 dry-run）。

In [ ]:
lab.reset(start_api=True)
lab.inject_inode_exhaustion()
_, files = lab.disk_usage()
out = lab.auto('inode_guard.py', '--apply')
print(out)

In [ ]:
lab.check('自动化处置返回 REMEDIATED', 'RESULT: REMEDIATED' in out)
print('\n自动化脚本已可作为 cron / 事件总线触发器复用')

## 复盘与思考题

1. 为什么删除大文件后 `df` 变化不明显，删除大量小文件却很明显？
2. XFS 与 ext4 的 inode 分配差异如何影响这类故障？
3. 如何设计会话文件存储，避免“一会话一文件”？

## 收尾

沙箱资源会在本 Cell 中停止；重新运行请从第 1 个代码 Cell 开始。


In [ ]:
lab.stop_all()
print('沙箱已停止')